# ZEUS：34 个 OpenML 数据集

使用 ZEUS 官方 `openml_ids`、数据加载函数和整表推理方式。每个 ID 保存一份 `data/embeddings/openml_<id>.npz`；两条 K-means 基线汇总到 `results/zeus_openml_baseline.csv`。先处理 Iris（补零）和 mfeat-fourier（PCA），再处理其余数据集。

官方源码提交：`3f75d280fe7edbe12fb1a80b4058b010af115e65`；权重：`models/zeus.pt`（README 链接，2026-09-29 下载）。簇数由真实标签给定，只用于模型的 `k`、K-means 的簇数及 ARI 计算。ARI 是单个随机种子结果，不是论文的多种子复现。

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import MinMaxScaler

root = Path.cwd().parent  # 在 notebooks/ 目录启动内核
sys.path.insert(0, str(root / 'vendor' / 'zeus'))
from zeus.configs import GMMConfig
from zeus.datasets import load_real_datasets
from zeus.initialziation import model_initialization
from zeus.utils import openml_ids

seed = 42
torch.manual_seed(seed)
config = GMMConfig(device='cpu', dist_based_logit=True)
model = model_initialization(config)
checkpoint = torch.load(root / 'models' / 'zeus.pt', map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['model'], strict=False)
model.eval()
print(f'{len(openml_ids)} 个 ID; 设备: {config.device}; 权重轮次: {checkpoint["epoch"]}')

34 个 ID; 设备: cpu; 权重轮次: 160


In [2]:
def process_dataset(openml_id):
    # 官方加载函数负责缺失值、类别编码和原始数值特征缩放。
    features, labels, _, _ = load_real_datasets(
        config.dim, openml_idx=[openml_id], return_whole_dataset=True
    )[0]
    n_samples, processed_dim = features.shape
    n_clusters = len(torch.unique(labels))

    # 与官方 evaluate_model 相同：超过 30 维做 PCA 后缩放，否则补零。
    if processed_dim > config.dim:
        reduced = PCA(n_components=config.pca_dim).fit_transform(features)
        reduced = MinMaxScaler(feature_range=(-1, 1)).fit_transform(reduced)
        model_features = torch.tensor(reduced, dtype=torch.float32)
    else:
        model_features = torch.cat(
            (features, torch.zeros(n_samples, config.dim - processed_dim)), dim=1
        )

    with torch.inference_mode():
        output = model(model_features.unsqueeze(1).to(config.device), k=n_clusters)
    embeddings = output[:-config.num_gaussians, 0, :].cpu().numpy()
    embeddings_scaled = MinMaxScaler(feature_range=(-1, 1)).fit_transform(embeddings)
    y = labels.numpy()
    assert embeddings.shape == (n_samples, config.embed_dim)
    assert np.isfinite(embeddings).all()

    path = root / 'data' / 'embeddings' / f'openml_{openml_id}.npz'
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path, openml_id=openml_id, embeddings=embeddings,
        embeddings_scaled=embeddings_scaled, labels=y,
        sample_indices=np.arange(n_samples), processed_features=features.numpy(),
    )

    scores = []
    for method, data in [('features', features.numpy()), ('scaled_zeus', embeddings_scaled)]:
        predictions = KMeans(n_clusters=n_clusters, n_init=100, random_state=seed).fit_predict(data)
        scores.append({
            'openml_id': openml_id, 'method': method, 'n_samples': n_samples,
            'processed_dim': processed_dim, 'model_input_dim': model_features.shape[1],
            'embedding_dim': embeddings.shape[1], 'n_clusters': n_clusters,
            'cluster_count_source': 'true labels', 'seed': seed, 'n_init': 100,
            'source_commit': '3f75d280fe7edbe12fb1a80b4058b010af115e65',
            'weight_id': f'zeus.pt epoch {checkpoint["epoch"]}',
            'ari': adjusted_rand_score(y, predictions),
        })
    return scores

In [3]:
ids = [61, 14] + [openml_id for openml_id in openml_ids if openml_id not in (61, 14)]
results_path = root / 'results' / 'zeus_openml_baseline.csv'
results_path.parent.mkdir(parents=True, exist_ok=True)
rows = []
for i, openml_id in enumerate(ids, start=1):
    scores = process_dataset(openml_id)
    rows.extend(scores)
    pd.DataFrame(rows).to_csv(results_path, index=False)
    print(f'{i:2}/{len(ids)}  ID {openml_id:5}  n={scores[0]["n_samples"]:4}  dim={scores[0]["processed_dim"]:3}  '          f'ARI: features={scores[0]["ari"]:.3f}, ZEUS={scores[1]["ari"]:.3f}', flush=True)

results = pd.DataFrame(rows)
print(f'已保存 {len(ids)} 个数据集、{len(results)} 条基线结果：{results_path.relative_to(root)}')

 1/34  ID    61  n= 150  dim=  4  ARI: features=0.716, ZEUS=0.851


 2/34  ID    14  n=2000  dim= 76  ARI: features=0.468, ZEUS=0.506


 3/34  ID    15  n= 699  dim=  9  ARI: features=0.834, ZEUS=0.813


 4/34  ID    16  n=2000  dim= 64  ARI: features=0.590, ZEUS=0.740


 5/34  ID    18  n=2000  dim=  6  ARI: features=0.556, ZEUS=0.516


...

31/34  ID 42261  n= 150  dim=  4  ARI: features=0.716, ZEUS=0.851


32/34  ID 42585  n= 344  dim= 10  ARI: features=0.540, ZEUS=0.951


33/34  ID  1462  n=1372  dim=  4  ARI: features=0.020, ZEUS=0.923


34/34  ID    51  n= 294  dim= 25  ARI: features=0.383, ZEUS=0.437


已保存 34 个数据集、68 条基线结果：results/zeus_openml_baseline.csv
